# Hadoop

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/hadoop.ipynb)

Apache Hadoop is the distributed storage and batch-processing foundation the modern data platform grew from: HDFS for durable, replicated block storage, YARN for scheduling, and MapReduce as the first programming model. The questions below cover HDFS mechanics, MapReduce internals, and what each idea became in modern engines.

## Architecture at a glance

```text
HDFS          - distributed storage: NameNode, DataNodes, blocks, replicas
YARN          - cluster resource manager: ResourceManager, NodeManagers, containers
MapReduce     - batch execution: Mapper, Reducer, shuffle and sort
Hadoop Common - shared libraries, RPC, configuration, filesystem abstraction
```

Mental model: HDFS is the storage layer, YARN is the compute broker, and MapReduce is one application that runs on YARN. The client talks to the NameNode for metadata and to the ResourceManager to launch an ApplicationMaster, which then negotiates containers on NodeManagers.

```xml
<!-- core-site.xml: which filesystem is the default -->
<property><name>fs.defaultFS</name><value>hdfs://nn1:8020</value></property>
```

Common mistake: treating HDFS and YARN as one daemon. They fail independently, so "cluster down" usually means one specific service, not all of Hadoop.

## HDFS

### How HDFS stores and serves a file

HDFS stores large files across a cluster by splitting them into blocks and replicating those blocks for fault tolerance. It is optimized for high-throughput reads and writes of large datasets, not low-latency updates to many small files.

HDFS gives one logical namespace over many disks: the client writes a large file, it is cut into blocks, and each block is placed on several DataNodes. Reads stream from the nearest replica, which is why throughput scales with the number of disks rather than the size of one disk.

```bash
hdfs dfs -stat "%n %o %r %b" /data/orders/part-00000   # name, blocksize, replication, size
```

Common mistake: expecting random updates. HDFS appends at the end of a file and rewrites the rest; it is not a POSIX filesystem.

The client asks the NameNode for block locations over RPC, then reads or writes block data directly to DataNodes over the data-transfer protocol. The NameNode is never in the data path.

```bash
hdfs getconf -namenodes
```

Common mistake: thinking all data flows through the NameNode; it only serves metadata, which is why a large namespace slows it down but bulk throughput does not.

HDFS blocks are typically 128 MB and are the unit of replication and placement; a traditional filesystem block is around 4 KB and is the unit of allocation. A large HDFS block amortizes NameNode metadata and lets a client stream without seeking.

Common mistake: comparing them as if both were disk-level allocation units. The HDFS block is a distributed abstraction.

```bash
# HDFS shell commands that come up in almost every interview
hdfs dfs -mkdir -p /data/orders/ds=2026-10-04
hdfs dfs -put ./orders.json /data/orders/ds=2026-10-04/
hdfs dfs -ls -h /data/orders/ds=2026-10-04/
hdfs dfs -du -s -h /data/orders
hdfs dfs -cat /data/orders/ds=2026-10-04/part-00000 | head -5
hdfs dfs -get /data/orders/ds=2026-10-04/part-00000 ./local.json
hdfs dfs -rm -r -skipTrash /data/orders/ds=2026-10-04/
```

### Replication and rack awareness

Core mechanics: a replica is one complete copy of a block on one DataNode, so replication 3 means three independent copies placed across failure domains. Durability grows with the count, and so does storage cost, which is why cold data is often moved to a lower factor or to erasure coding.

```bash
hdfs dfs -setrep -w 2 /data/cold
hdfs dfs -stat "%b %r" /data/cold/part-00000
```

Common mistake: changing `dfs.replication` in `hdfs-site.xml` and expecting existing files to change; only new files pick up the default.

Placement policy in detail: the first replica goes on the writer's node (or a random node if the writer is outside the cluster), the second on a node in a different rack, and the third on a node in the same rack as the second. Reads prefer the replica at the smallest topology distance.

```bash
hdfs dfsadmin -printTopology
hdfs fsck /data -files -blocks -locations | head
```

Common mistake: one rack per node when racks are physically shared; placement then spreads across what is really the same failure domain.

The client writes to a pipeline of three DataNodes; packets flow down the pipeline and acknowledgements flow back up. If a node fails mid-write, the pipeline is rebuilt with the remaining nodes and a new replica, and the NameNode records the change.

Common mistake: thinking the client waits for all three acks per packet; it waits for the pipeline ack, which is generated as packets propagate.

The block has no other copy, so the file is lost once the NameNode marks that DataNode dead and removes its block locations. `hdfs fsck` reports missing blocks and clients get `BlockMissingException`.

```bash
hdfs fsck /data/fragile -files -blocks
```

Common mistake: setting replication 1 for an ingested-only dataset "because we can replay it", then discovering the replay source is gone too.

The cell below simulates the default placement policy and shows that the last block holds only the remainder.

In [ ]:
import math
import random

MB = 1024**2
BLOCK_SIZE = 128 * MB
TOPOLOGY = {"dn1": "/rack1", "dn2": "/rack1", "dn3": "/rack1", "dn4": "/rack2", "dn5": "/rack2", "dn6": "/rack3"}


def place_block(writer, block_id):
    """Default policy: writer's node, a node on another rack, then another node on that remote rack."""
    rng = random.Random(block_id)
    nodes = sorted(TOPOLOGY)
    second = rng.choice([n for n in nodes if TOPOLOGY[n] != TOPOLOGY[writer]])
    same_remote_rack = [n for n in nodes if TOPOLOGY[n] == TOPOLOGY[second] and n != second]
    third = rng.choice(same_remote_rack or [n for n in nodes if n not in (writer, second)])
    return [writer, second, third]


def write_file(size_bytes, writer="dn1", replication=3):
    blocks = []
    for block_id in range(math.ceil(size_bytes / BLOCK_SIZE)):
        length = min(BLOCK_SIZE, size_bytes - block_id * BLOCK_SIZE)
        blocks.append({"id": block_id, "bytes": length, "replicas": place_block(writer, block_id)[:replication]})
    return blocks


blocks = write_file(300 * MB)
for block in blocks:
    racks = {TOPOLOGY[node] for node in block["replicas"]}
    print(block["id"], block["bytes"] // MB, "MB ->", block["replicas"], sorted(racks))
    assert len(set(block["replicas"])) == 3 and len(racks) >= 2  # survives a whole-rack failure

# Blocks are a maximum size, not an allocation unit: the last block holds only the remainder.
physical = sum(block["bytes"] * len(block["replicas"]) for block in blocks)
assert [block["bytes"] // MB for block in blocks] == [128, 128, 44]
assert physical == 300 * MB * 3
small_file = write_file(25 * MB)
assert sum(b["bytes"] * len(b["replicas"]) for b in small_file) == 75 * MB  # not 3 x 128 MB
print("raw disk for 300 MB file:", physical // MB, "MB; for 25 MB file:", 75, "MB")

### How much disk does a 25 MB file use?

One block entry in the NameNode, but only 25 MB on each DataNode: the block size is a maximum, not an allocation unit, so a replica's local file is exactly as large as its data. With replication 3 the file uses 75 MB of raw disk. The real cost of a small file is the NameNode object and the map task it creates.

Common mistake: answering 3 x 128 MB = 384 MB. HDFS does not pad blocks.

### Heartbeats, block reports and DataNode failure

Heartbeats (every 3 seconds by default) prove liveness, and block reports (every 6 hours, `dfs.blockreport.intervalMsec`) list every block the DataNode holds. The NameNode marks a node dead after `dfs.namenode.heartbeat.recheck-interval` plus 10 heartbeat intervals, then re-replicates its blocks.

```xml
<property><name>dfs.heartbeat.interval</name><value>3</value></property>
<property><name>dfs.blockreport.intervalMsec</name><value>21600000</value></property>
```

Common mistake: thinking the NameNode polls DataNodes. DataNodes push both messages.

It stops receiving heartbeats, marks the node dead after the timeout, removes its replicas from the block map, and queues the affected blocks for re-replication from surviving replicas. If a block loses all replicas it is reported missing.

```bash
hdfs fsck / -list-corruptfileblocks
```

Common mistake: lowering the heartbeat timeout too far, which flaps nodes in and out and causes unnecessary re-replication.

### NameNode metadata, checkpoints and HA

Checkpoint mechanics: every namespace change is appended to the edit log, and periodically the fsimage plus the edits are merged into a new fsimage. A long edit log means slow restarts and more work at checkpoint time, so `dfs.namenode.checkpoint.period` and `checkpoint.txns` bound the window.

```bash
hdfs dfsadmin -rollEdits
hdfs dfsadmin -saveNamespace
```

Common mistake: letting the edits directory grow unbounded on a busy namespace without a working checkpoint node.

It is not a standby. It periodically downloads the fsimage and edit log, merges them into a new fsimage, and uploads that checkpoint back, which keeps the edit log short and restart times bounded. In Hadoop 3 the same role is the Checkpoint node, usually run on the standby NameNode.

Common mistake: calling it a backup NameNode. If the NameNode dies, the Secondary does not take over.

Without HA the cluster stops serving writes and the namespace is gone unless you have an fsimage checkpoint and can replay edits; DataNodes keep serving block bytes for a while but no client can find them by path. With HA, the standby is promoted and clients fail over.

Common mistake: backing up only the fsimage. You need the fsimage and the last edit logs together, or the namespace is incomplete.

HA closes the edit-log gap: the active NameNode writes edits to a shared JournalNode quorum and the standby tails the same log, so it can take over in seconds. Federation splits the namespace into independent namespaces, each with its own active/standby pair, because one NameNode's heap limits total file count.

```xml
<property><name>dfs.nameservices</name><value>ns1,ns2</value></property>
<property><name>dfs.ha.automatic-failover.enabled</name><value>true</value></property>
<property><name>dfs.journalnode.edits.dir</name><value>/journal</value></property>
```

Common mistake: thinking HA removes the size limit. It removes the single point of failure; only federation scales the namespace.

Hadoop 1 had a single NameNode with the Secondary NameNode for checkpoints and no automatic failover. Hadoop 2 introduced HA (active/standby with JournalNodes), federation for multiple namespaces, and separated resource management into YARN.

Common mistake: saying Hadoop 2 removed the Secondary NameNode; it was renamed and folded into the standby/checkpoint role.

### Block scanner, fsck, safemode and the balancer

Core mechanics: a file is cut into blocks of `dfs.blocksize` (128 MB default), and each block is an independent unit of placement, so its replicas can live on different nodes. The DataNode block scanner periodically verifies stored blocks against their checksums and reports corruption to the NameNode, which re-replicates from a healthy copy.

```bash
hdfs fsck /data/orders -files -blocks
```

Common mistake: assuming the scanner runs on demand. It is a background pass throttled by `dfs.block.scanner.volume.bytes.per.second`.

```bash
hdfs fsck /data/orders -files -blocks -locations   # file and block report
hdfs fsck /data/orders -list-corruptfileblocks     # names of corrupt files
hdfs fsck /data/orders -move                       # move corrupt blocks to /lost+found
hdfs fsck /data/orders -delete                     # delete corrupt replicas
hdfs fsck /data/orders -openforwrite               # files left open by failed writers
```

The threshold matters: the NameNode leaves safemode only when `dfs.namenode.safemode.threshold-pct` of blocks have been reported with at least `dfs.namenode.replication.min` replicas, plus an extension window. During that time writes and appends are rejected with a safemode exception.

```xml
<property><name>dfs.namenode.safemode.threshold-pct</name><value>0.999</value></property>
```

Common mistake: forcing `-safemode leave` on a large restart and then seeing a storm of re-replication for blocks that had simply not reported yet.

The balancer moves blocks between DataNodes to even out disk usage, within a threshold (`dfs.balance.bandwidthPerSec`, `dfs.datanode.balance.bandwidthPerSec`). It does not help when the imbalance is caused by one full disk on an otherwise even node.

```bash
hdfs balancer -threshold 5 -policy datanode
```

Common mistake: running the balancer with an unlimited bandwidth cap; it saturates the network and starves real jobs.

### Small files

Every file, directory and block costs roughly 150 bytes of NameNode heap, and every file becomes at least one map task. A million 10 KB files is about 150 MB of heap for almost no data, plus a million tasks that each pay JVM and scheduling overhead. The fix is compaction into block-sized or larger columnar files (for example many 1 MB JSON files into a few 256 MB Parquet files), ideally on ingest.

```bash
hdfs dfs -count -q -h /data/logs/2026/10/04            # files, dirs, bytes, quota
hadoop archive -archiveName logs.har -p /data/logs/2026/10/04 /data/archive   # HAR: packs files, keeps format
hdfs dfsadmin -setQuota 500000 /data/events            # stop a path flooding the namespace
```

Common mistake: measuring only raw bytes. The real cost is objects, not gigabytes, and concatenating gzip files does not make them splittable.

### Erasure coding and caching

Erasure coding trades CPU for storage: RS-6-3 stores 9 fragments for 6 data units, a 1.5x overhead instead of 3x for replication, but reads fetch and decode multiple fragments, so it suits cold, large, rarely read data. HDFS caching (`hdfs cacheadmin`) is the opposite lever, pinning hot blocks in DataNode off-heap memory for repeated reads.

```bash
hdfs ec -enablePolicy -policy RS-6-3-1024k
hdfs ec -setPolicy -path /data/cold -policy RS-6-3-1024k
hdfs cacheadmin -addPool hot_pool -limit 200g
hdfs cacheadmin -addDirective -path /data/hot -pool hot_pool -replication 1
```

Common mistake: applying EC to data that is rewritten frequently, or caching a directory larger than the pool.

## MapReduce

### The core idea

Move the computation to the data instead of the data to the computation. A job is split into tasks that are scheduled onto the nodes that already hold the blocks, so only intermediate results cross the network.

That single idea explains the rest: block placement, data locality in scheduling, and the shuffle being the expensive part of a job.

Because moving a 128 MB block over the network is far more expensive than moving the few KB of task code to the node that already has the block. Colocating them gives data locality for free and the network carries only results.

Common mistake: splitting storage and compute in a way that forces every task to read remotely, which turns the network into the bottleneck.

### Anatomy of a job

A job is submitted as one or more map tasks over input splits, zero or more reduce tasks over partitions, plus a client-side driver and a per-application ApplicationMaster. The AM requests containers, tracks task attempts, and retries failures.

```bash
yarn jar hadoop-mapreduce-examples.jar wordcount /input /output
```

Common mistake: thinking the client runs the job. The client submits and polls; the ApplicationMaster does the work.

```java
Configuration conf = new Configuration();
Job job = Job.getInstance(conf, "orders-etl");
job.setJarByClass(OrdersJob.class);
job.setMapperClass(OrdersMapper.class);
job.setCombinerClass(OrdersCombiner.class);
job.setPartitionerClass(RegionPartitioner.class);
job.setReducerClass(OrdersReducer.class);
job.setOutputKeyClass(Text.class);
job.setOutputValueClass(IntWritable.class);
FileInputFormat.addInputPath(job, new Path(args[0]));
FileOutputFormat.setOutputPath(job, new Path(args[1]));
job.setNumReduceTasks(8);
System.exit(job.waitForCompletion(true) ? 0 : 1);
```

In Hadoop 1 the JobTracker scheduled every task and tracked every TaskTracker. In Hadoop 2+ that role is split: the ResourceManager allocates containers and each application's ApplicationMaster schedules its own tasks, which removes the single job bottleneck.

Common mistake: mixing generations in an answer. JobTracker/TaskTracker are Hadoop 1 terms; ResourceManager/NodeManager/ApplicationMaster are Hadoop 2 and 3.

A mapper works on one input split in isolation and sees one record at a time; a reducer works on one partition and sees every value for a key as an iterable. Mappers scale with input size, reducers scale with key cardinality and skew.

Common mistake: assuming values arrive at the reducer in any particular order; only the keys are sorted.

### Word count, end to end

A MapReduce job in a few lines of Python: map each split, optionally combine, partition by key hash, sort and group per reducer, then reduce. Hadoop adds distribution, spill-to-disk and retries, but the data flow is exactly this.

In [ ]:
from collections import Counter, defaultdict
from itertools import groupby
from operator import itemgetter
import zlib

SPLITS = [
    "the quick brown fox jumps over the lazy dog",
    "the dog barks and the fox runs",
    "a lazy afternoon for the brown dog",
]
NUM_REDUCERS = 3


def mapper(line):
    for word in line.split():
        yield word, 1


def combiner(pairs):
    totals = Counter()
    for word, count in pairs:
        totals[word] += count
    return sorted(totals.items())


def partition(key, num_reducers):
    # Stable hash: Python's hash() is salted per process. Hadoop uses (key.hashCode() & MAX_INT) % n.
    return zlib.crc32(key.encode()) % num_reducers


def reducer(key, values):
    yield key, sum(values)


def run_job(splits, num_reducers, use_combiner=True):
    partitions = defaultdict(list)  # reducer id -> fetched map output (the shuffle)
    shuffled_records = 0
    for split in splits:  # one map task per input split
        pairs = list(mapper(split))
        if use_combiner:
            pairs = combiner(pairs)
        shuffled_records += len(pairs)
        for key, value in pairs:
            partitions[partition(key, num_reducers)].append((key, value))
    output = {}
    for reducer_id in range(num_reducers):
        records = sorted(partitions[reducer_id], key=itemgetter(0))  # merge-sort by key
        for key, group in groupby(records, key=itemgetter(0)):
            for out_key, out_value in reducer(key, (value for _, value in group)):
                output[out_key] = out_value
    return output, shuffled_records


with_combiner, shuffled_with = run_job(SPLITS, NUM_REDUCERS)
without_combiner, shuffled_without = run_job(SPLITS, NUM_REDUCERS, use_combiner=False)

assert with_combiner == without_combiner  # a combiner changes cost, never the answer
assert with_combiner["the"] == 5 and with_combiner["dog"] == 3
assert shuffled_with < shuffled_without
print("top words:", sorted(with_combiner.items(), key=lambda kv: (-kv[1], kv[0]))[:4])
print("records shuffled without / with combiner:", shuffled_without, "/", shuffled_with)

Streaming lets any executable read the map input as lines on stdin and write `key<TAB>value` lines on stdout. Hadoop ships the mapper and reducer scripts through the distributed cache and runs them as the mapper and reducer.

```bash
yarn jar hadoop-streaming.jar \
  -D mapreduce.job.reduces=4 \
  -files mapper.py,reducer.py \
  -mapper mapper.py -reducer reducer.py \
  -input /data/words -output /data/out
```

Common mistake: forgetting that the default separator is a tab, not a space.

The streaming contract in plain Python, run as a local pipeline:

In [ ]:
def streaming_mapper(lines):
    """mapper.py: read lines on stdin, write 'word<TAB>1'."""
    for line in lines:
        for word in line.strip().split():
            yield f"{word}\t1"


def streaming_reducer(sorted_lines):
    """reducer.py: input arrives sorted by key, so a running total per key is enough."""
    current_word, current_count = None, 0
    for line in sorted_lines:
        word, count = line.split("\t", 1)
        if word == current_word:
            current_count += int(count)
        else:
            if current_word is not None:
                yield f"{current_word}\t{current_count}"
            current_word, current_count = word, int(count)
    if current_word is not None:
        yield f"{current_word}\t{current_count}"


# Same pipeline as: cat input | python3 mapper.py | sort | python3 reducer.py
streamed = {}
for line in streaming_reducer(sorted(streaming_mapper(SPLITS))):
    word, count = line.split("\t")
    streamed[word] = int(count)

assert streamed == with_combiner
print("streaming output matches the engine:", len(streamed), "distinct words")

### Input splits versus blocks

A block is a physical unit of storage with a fixed size and replicas. A split is a logical unit of work computed by the InputFormat, possibly combining several blocks or splitting one. Blocks live on DataNodes; splits live in the job's scheduling plan.

Common mistake: reporting "number of blocks" as "number of mappers". With small files or a non-splittable codec they diverge.

Split sizing arithmetic: for a file of size S with split size B, the split count is `ceil(S / B)` (the last split holds the remainder), and split size defaults to the block size. Setting `split.maxsize` smaller increases map count and shortens each map.

```text
10 GB file, 128 MB split  -> ceil(10*1024/128) = 80 maps
10 GB file, 256 MB split  -> ceil(10*1024/256) = 40 maps
```

Common mistake: choosing a split size that makes each map finish in seconds; scheduling overhead then dominates the job.

`InputFormat.isSplitable` tells the framework whether one file can be divided into several splits. `FileInputFormat` returns true, but gzip, bzip2, and most archive formats override it to false, forcing one mapper per file.

```java
protected boolean isSplitable(JobContext context, Path filename) { return false; }
```

Common mistake: shipping gzip for very large files. A 100 GB gzip file becomes one mapper unless you use a splittable format such as bzip2 or an indexed container format.

### Partitioners and skew

Subclass `Partitioner<K,V>`, return an index in `[0, numPartitions)`, and register it with `job.setPartitionerClass`. Make sure the class is on the task classpath, usually via the job jar.

```java
public class SkewPartitioner extends Partitioner<Text, Text> {
  public int getPartition(Text k, Text v, int numPartitions) {
    if (k.toString().equals("hot")) return 0;
    return (k.hashCode() & Integer.MAX_VALUE) % (numPartitions - 1) + 1;
  }
}
```

Common mistake: returning a value outside the range, which fails the task with an `IllegalArgumentException`.

Skew in action: one tenant owns half the records, so hash partitioning sends half the work to one reducer. Salting spreads that key and a second pass re-aggregates.

In [ ]:
from statistics import mean

HOT_KEYS = {"hot_tenant"}
SALT_BUCKETS = 8
REDUCERS = 8
records = ["hot_tenant"] * 4_000 + [f"tenant_{i}" for i in range(1_000) for _ in range(4)]


def salted(key, record_index):
    """Spread only the known hot key over SALT_BUCKETS sub-keys."""
    return f"{key}#{record_index % SALT_BUCKETS}" if key in HOT_KEYS else key


def reducer_loads(keys):
    loads = Counter(partition(key, REDUCERS) for key in keys)
    return [loads[r] for r in range(REDUCERS)]


plain = reducer_loads(records)
salted_keys = [salted(key, i) for i, key in enumerate(records)]
balanced = reducer_loads(salted_keys)
skew_plain = max(plain) / mean(plain)
skew_salted = max(balanced) / mean(balanced)

# Salting needs a second aggregation: stage 1 counts per salted key, stage 2 strips the salt.
partial = Counter(salted_keys)
final = Counter()
for salted_key, count in partial.items():
    final[salted_key.split("#")[0]] += count

assert final == Counter(records)
assert skew_salted < skew_plain
print("reducer loads, hash partitioner:", plain, f"max/mean={skew_plain:.2f}")
print("reducer loads, salted hot key:  ", balanced, f"max/mean={skew_salted:.2f}")

### Combiners

Correctness test: a combiner is safe when the reducer's operation is commutative and associative, so `sum`, `max`, `min` and `count` are fine while `average` is not. It runs on each spill and again during the map-side merge, so the number of invocations is not fixed.

```xml
<property><name>mapreduce.map.combine.minspills</name><value>3</value></property>
```

Common mistake: assuming the combiner cuts shuffle bytes by a fixed factor; the reduction depends on how many duplicate keys appear within one mapper's output.

```text
Safe as a combiner : sum, max, min, count (and average if you emit sum+count)
Unsafe as a combiner: median, exact distinct count, order-dependent logic
```

### Shuffle and sort

The map side sorts each spill and merges spills into one run per partition on local disk; the reduce side fetches those runs over HTTP and merges them. Sorting therefore happens twice, once on the map output and once during the merge, not in a central place.

```bash
mapred job -history /output | grep -i shuffle
```

Common mistake: thinking shuffle is a network-only step. Map-side spill and merge are disk-bound and often dominate.

In the map task's local temporary directory on the node's data disk, controlled by `mapreduce.cluster.local.dir` and `mapreduce.task.io.sort.mb`. It is never on HDFS.

Common mistake: sizing only HDFS when planning disk; shuffle spool can need as much local disk as the map output.

Reduce progress has three parts: shuffle, sort and reduce. Shuffle can start as soon as some map outputs exist, so the reduce phase reports progress while maps are still running. The scheduler may also start reducers early with a fraction of their inputs.

```xml
<property><name>mapreduce.job.reduce.slowstart.completedmaps</name><value>0.05</value></property>
```

Common mistake: assuming reducers wait for 100 percent of maps; the default slowstart begins fetching after five percent.

### Map-side versus reduce-side joins

A map-side join loads the small table into memory (or the distributed cache) and enriches each map record locally, with no shuffle. A reduce-side join partitions both datasets by the join key and joins groups in the reducer, which handles large-large joins at the cost of a full shuffle.

Common mistake: a map-side join on a table that does not fit in the container's heap; `mapreduce.map.memory.mb` is the ceiling.

In [ ]:
customers = {1: "US", 2: "DE", 3: "IN"}  # small dimension: fits in every mapper's memory
orders = [(101, 1, 30.0), (102, 2, 12.5), (103, 1, 7.5), (104, 3, 99.0), (105, 4, 5.0)]

# Map-side (broadcast) join: look up the in-memory table per record; no shuffle at all.
map_side = [(order_id, customers[cid], amount) for order_id, cid, amount in orders if cid in customers]

# Reduce-side join: tag both inputs, shuffle by join key, join within each key group.
tagged = [(cid, "C", country) for cid, country in customers.items()]
tagged += [(cid, "O", (order_id, amount)) for order_id, cid, amount in orders]
reduce_side = []
for cid, group in groupby(sorted(tagged, key=itemgetter(0, 1)), key=itemgetter(0)):
    rows = list(group)
    countries = [payload for _, tag, payload in rows if tag == "C"]
    for _, tag, payload in rows:
        if tag == "O":
            reduce_side.extend((payload[0], country, payload[1]) for country in countries)

assert sorted(map_side) == sorted(reduce_side)
assert all(order_id != 105 for order_id, _, _ in map_side)  # inner join drops the unmatched order
print(sorted(map_side))

### Map-only jobs, speculative execution and output commit

The job runs map-only, the shuffle and sort are skipped, and each map task writes its own part file. No global aggregation happens, so operations that need all values for a key are not possible.

```java
job.setNumReduceTasks(0);
```

Common mistake: expecting a single output file. Map-only output has one part per split, so you get many small files.

The AM tracks task progress and starts a duplicate attempt of a straggler on another node; whichever finishes first wins and the other is killed. It helps with a slow disk or a noisy neighbor, and hurts when the whole cluster is oversubscribed.

```xml
<property><name>mapreduce.map.speculative</name><value>true</value></property>
<property><name>mapreduce.reduce.speculative</name><value>true</value></property>
```

Common mistake: leaving speculation on for a non-idempotent task with external side effects; the duplicate may run concurrently with the original.

The OutputCommitter decides when output becomes visible: `FileOutputCommitter` writes task output to a temporary directory and renames it into place once the job succeeds. It also handles task commit and abort on failure.

```java
job.setOutputCommitter(FileOutputCommitter.class);
```

Common mistake: reading from the output directory of a running job; the `_temporary` files are not the final result and the `_SUCCESS` marker is missing.

Keys must implement `WritableComparable` so they can be serialized and sorted; values must implement `Writable`. Keys also need a stable `hashCode` consistent with `compareTo`, because the partitioner relies on it.

Common mistake: a mutable key whose fields change after it is written; the sort and partition decisions are taken on stale values.

### How many mappers and reducers run

Mappers are fixed by the number of input splits. Reducers are set by `mapreduce.job.reduces` and constrained by available containers. Concurrency is limited by total cluster slots and by `yarn.nodemanager.resource.memory-mb` per node.

Common mistake: setting 1000 reducers on a cluster with 100 containers; the last reducers start only after earlier ones finish, so wall-clock time is unchanged.

```text
Containers per node = min(yarn.nodemanager.resource.memory-mb / task memory,
                         yarn.nodemanager.resource.cpu-vcores)
On a 96 GB / 32 vcore node with 2 GB map tasks, roughly 32 maps can run
concurrently, bounded by vcores.
```

### Where MapReduce is a poor fit

```text
Good fit    : one-pass batch scans, huge inputs, fault-tolerant batch ETL
Poor fit    : iterative ML, interactive queries, graph algorithms, streaming,
              low-latency lookups
Spark, Tez or Flink replace MapReduce when the same data is revisited many
times, because they keep intermediate state in memory instead of on disk.
```

## Operating and tuning

### Configuration files and precedence

Precedence is property-level, not file-level: a value in the job configuration beats `mapred-site.xml`, which beats `mapred-default.xml`. `-D key=value` on the command line is the highest precedence.

```bash
hadoop jar job.jar Main -D mapreduce.job.reduces=16
```

Common mistake: assuming adding a file replaces all defaults; only the keys you set are overridden.

### Diagnosing a slow job

Start with counters, not guesswork. `Map input records`, `Map output bytes`, `Combine input/output records`, `Reduce shuffle bytes`, `Spilled Records`, `CPU time spent` and `Physical memory snapshot` tell you whether time goes to reading, shuffling, spilling or user code. The JobHistoryServer keeps the same data after the job ends.

```bash
mapred job -history /output
mapred job -history all /output | grep -iE "shuffle|spill|combine"
```

Common mistake: tuning memory first. A job that is 85 percent shuffle and spill will not speed up from a bigger map heap.

The classic signature of skew is all reducers finishing while a few sit at 99 percent for hours. Confirm it from the per-reducer counts in the history output, then fix it by salting the hot key, using a custom partitioner, or splitting the hot value into its own pipeline.

```text
Reduce input groups: reducer 0 = 12,418,002 ; reducers 1-31 < 40,000
=> one key dominates reducer 0
```

Common mistake: adding reducers. More reducers do not help when one key must be processed by one reducer.

Worked diagnosis for a word-count-like job: the job took 3 hours, maps took 6 minutes, and shuffle bytes were 900 GB. That points at map output size and spill, so add a combiner, compress the map output, and raise the sort buffer within the container's heap before touching the reducers.

```xml
<property><name>mapreduce.map.output.compress</name><value>true</value></property>
<property><name>mapreduce.map.output.compress.codec</name><value>org.apache.hadoop.io.compress.SnappyCodec</value></property>
```

Common mistake: reading only the wall-clock total. Split it into map time, shuffle time and reduce time from the history counters.

### Split sizing and compression

Map count comes from splits, and split size defaults to the block size. `mapreduce.input.fileinputformat.split.maxsize` caps a split, `split.minsize` floors it, and the effective size for a file that is not splittable is the whole file. Aim for maps long enough to amortize startup but numerous enough to fill the cluster.

```xml
<property><name>mapreduce.input.fileinputformat.split.maxsize</name><value>268435456</value></property>
<property><name>dfs.blocksize</name><value>268435456</value></property>
```

Common mistake: changing `dfs.blocksize` for old data and expecting new splits; the InputFormat reads block locations, so a rewrite is needed to re-block.

Compression trade-off: gzip compresses well but is not splittable, so one large `.gz` file becomes one mapper; bzip2 is splittable but slow; Snappy and LZ4 are fast and splittable, and columnar formats such as Parquet and ORC carry their own splittable blocks with per-row-group statistics.

```bash
hdfs dfs -D dfs.blocksize=134217728 -put ./events.json /data/events/
hadoop jar job.jar Main -D mapreduce.input.fileinputformat.split.maxsize=67108864
```

Common mistake: choosing the smallest file size on disk rather than the fastest decode for a shuffle-heavy job.

Worked comparison: 400 GB/day of JSON at 8:1 gzip gives 50 GB stored but one mapper per file, while Parquet with Snappy gives roughly 120 GB stored and is splittable per row group, so both scan parallelism and downstream projection improve.

```text
JSON 400 GB -> gzip 50 GB, 1 map per file, full-file decode
Parquet + Snappy ~120 GB, N maps per file, column pruning
```

Common mistake: comparing only stored bytes; the decode CPU and the loss of split parallelism are usually the larger cost.

### Shuffle and reducer tuning

The map side writes to a circular buffer of `mapreduce.task.io.sort.mb` and sorts and spills at `mapreduce.map.sort.spill.percent`. Small buffers produce many spills and extra merge passes; the merge reads `mapreduce.task.io.sort.factor` runs at a time. All of this memory comes out of the task heap.

```xml
<property><name>mapreduce.task.io.sort.mb</name><value>512</value></property>
<property><name>mapreduce.map.sort.spill.percent</name><value>0.80</value></property>
<property><name>mapreduce.task.io.sort.factor</name><value>64</value></property>
```

Common mistake: setting `io.sort.mb` close to the container's `mapreduce.map.memory.mb`; the JVM heap must hold the buffer plus the user code.

The reduce side fetches with `mapreduce.reduce.shuffle.parallelcopies` and merges in memory until `mapreduce.reduce.shuffle.merge.percent` of `mapreduce.reduce.memory.mb`; beyond that it spills. If reducers report `Shuffled Maps = 0` for a long time, fetch or DNS issues, not memory, are the problem.

```xml
<property><name>mapreduce.reduce.shuffle.parallelcopies</name><value>20</value></property>
<property><name>mapreduce.reduce.shuffle.merge.percent</name><value>0.66</value></property>
```

Common mistake: raising parallel copies for a small network; the extra fetches contend with the mappers that are still writing.

A combiner plus map-output compression is the highest-leverage shuffle change: the combiner removes duplicate keys per mapper, and compression shrinks what remains. Set `mapreduce.map.output.compress=true` with a fast codec, and remember the combiner only helps when keys repeat inside one mapper.

```xml
<property><name>mapreduce.job.reduce.slowstart.completedmaps</name><value>0.05</value></property>
<property><name>mapreduce.map.output.compress</name><value>true</value></property>
```

Common mistake: running a combiner for a job with no duplicate keys; it adds CPU and changes nothing on the wire.

### YARN container sizing

YARN sizing starts at the node: `yarn.nodemanager.resource.memory-mb` and `yarn.nodemanager.resource.cpu-vcores` are what the NodeManager offers, and containers are multiples of `yarn.scheduler.minimum-allocation-mb` between the min and max. Queue policy then caps each queue with `yarn.scheduler.capacity.<queue>.capacity` and `maximum-capacity`.

```xml
<property><name>yarn.nodemanager.resource.memory-mb</name><value>98304</value></property>
<property><name>yarn.scheduler.minimum-allocation-mb</name><value>1024</value></property>
<property><name>yarn.scheduler.capacity.root.etl.capacity</name><value>60</value></property>
<property><name>yarn.scheduler.capacity.root.etl.maximum-capacity</name><value>80</value></property>
```

Common mistake: setting container requests below the minimum allocation; YARN rounds them up and the job uses more cluster memory than planned.

Rule of thumb for map and reduce containers: `mapreduce.map.memory.mb` covers the JVM heap plus overhead, and `mapreduce.map.java.opts` should be about 80 percent of it. The map sort buffer and the reduce merge buffer must fit inside that heap, or the container is killed rather than slowed.

```xml
<property><name>mapreduce.map.memory.mb</name><value>2048</value></property>
<property><name>mapreduce.map.java.opts</name><value>-Xmx1638m</value></property>
<property><name>mapreduce.reduce.memory.mb</name><value>4096</value></property>
<property><name>mapreduce.reduce.java.opts</name><value>-Xmx3276m</value></property>
```

Common mistake: leaving `mapreduce.admin.map.child.java.opts` defaults and then wondering why a 4 GB container still runs out of memory.

## Trade-offs: when to move off Hadoop

+ **Shuffle cost.** MapReduce and Hive both write intermediate data to disk between stages. A job with five stages reads and writes the shuffle data five times, so fewer, wider stages beat many narrow ones.
+ **Data skew.** A single hot key (a null `customer_id`, one huge tenant) lands on one reducer and turns a 20-minute job into a 4-hour job. Fix it with `hive.optimize.skewjoin`, salting the key, or splitting the hot value into its own pipeline.
+ **Small files.** Every file costs NameNode memory and one task. Compacting to 128-512 MB files is the single highest-leverage storage fix.
+ **Block size versus split size.** Maps equal splits, not blocks; a non-splittable codec (gzip) forces one mapper per file.
+ **Slot-based scheduling is coarse.** YARN allocates whole containers; a task asking for 2 GB on a 1.5 GB container fails with a container error rather than degrading gracefully.
+ **When to move off Hadoop/Hive.** Move to object storage plus Spark/Trino or a lakehouse table format when: most data is cold, clusters are idle most of the day, you need sub-minute dashboards, or the operational cost of an HDFS/YARN cluster exceeds the compute you actually use. Keep HDFS for very large, repeatedly scanned datasets that justify dedicated hardware.
+ **When Hadoop still wins.** Petabyte-scale batch ETL with predictable schedules, strict data residency, and a large installed Hive SQL base is often cheaper on dedicated YARN than paying cloud egress and per-query costs.

The short version: HDFS buys cheap sequential throughput at the cost of latency and update support, while YARN buys shared-cluster utilization at the cost of tuning queue policy and container sizing. Decide by access pattern and SLA, not by data volume alone.

```text
scan-heavy, append-only, batch SLA        -> HDFS + YARN is a good fit
random updates, sub-second, interactive   -> object store + Spark/Trino, or a warehouse
```

Common mistake: migrating because a competitor uses Spark, rather than because the access pattern changed.

## Things to remember

### Key numbers and limits

| Topic | Fact | Why it matters |
|---|---|---|
| HDFS blocks | files are split into large blocks (e.g. 128 MB), replicated 3x across racks by default | big sequential reads beat many small files |
| NameNode | holds all metadata (namespace + block map) in memory | the NameNode's RAM bounds file *count*, not bytes — the root of the small-files problem |
| Secondary NameNode | not a standby; it only checkpoints the edit log | NameNode HA (JournalNodes) is the real failover story |
| Heartbeats | DataNodes heartbeat every 3 seconds; block reports every ~hours per block | a missed heartbeat window marks a node stale/dead |
| Splits vs blocks | mappers count = input splits; a map task gets one split | file count and splittable compression decide parallelism |
| Shuffle | map output spills to local disk, sorted, then fetched by reducers | the shuffle is the expensive motion; combiners and fewer keys shrink it |
| Speculative execution | slow straggler tasks get duplicated on other nodes | great for skewed hardware, wasteful with side-effecting tasks |
| YARN vs MRv1 | JobTracker/TaskTracker (v1) became ResourceManager/NodeManager (YARN) | modern clusters run YARN; Spark and Hive on TeX/Spark ride it |

### Semantics to memorize

* **HDFS is write-once, read-many.** No in-place updates at an arbitrary offset — append or rewrite. It is a batch store for large sequential reads, not a random-access filesystem.
* **Data locality is the whole point.** Compute moves to the block; the map/reduce model exists to avoid dragging terabytes over the network. Storage and compute living on the same nodes is a consequence, not a coincidence.
* **Splits drive parallelism, not blocks directly.** The InputFormat computes splits; only some codecs (snappy-in-container, bzip2) are splittable, which is why gzip'd logs serialize a job onto one mapper.
* **The shuffle is where MR jobs die.** Spill, sort, and fetch dominate; skewed keys strand a few reducers while the rest finish.
* **Small files are a metadata tax.** Every file costs NameNode RAM and one map task slot at minimum — compact on ingest.
* **This is the model that spawned everything else.** Spark's stages, Hive's physical plans, and even modern lakehouse engines reuse these mechanics; learn them once here.

### Mistakes that keep coming back

* Landing millions of small files in HDFS then wondering why jobs crawl — compact into ~block-sized+ files; use SequenceFile/HAR or (better) a table format.
* gzipping inputs that must be split — a single non-splittable file means one mapper; use bzip2 or a splittable container, or recompress.
* Reducers = 0 by accident (default in some APIs) — the map output is the job output; explicit `1` or a real count.
* Treating MapReduce as a general engine — iterative algorithms (ML) re-read HDFS every pass; that use case is what Spark fixed.
* Assuming the Secondary NameNode provides failover — it never did; NameNode HA needs JournalNodes/ZKFC.
* Trusting speculative execution with side-effecting tasks — a duplicate task 'succeeds' twice.
* Sizing reducers from guesswork — reducer count should follow shuffle volume, and the skewed keys need salting, not more reducers.

## Real world use cases

### 1. Nightly batch ETL for a retail data platform

A retailer lands 8 TB of point-of-sale and clickstream data per day. The batch layer runs on YARN, reads JSON from HDFS, joins against dimension files held in the distributed cache, and writes partitioned, ORC or Parquet facts that downstream Hive and Spark jobs consume.

```bash
yarn jar etl.jar com.acme.DailyEtl \
  -D mapreduce.job.queuename=etl \
  -D mapreduce.job.reduces=200 \
  -D mapreduce.map.memory.mb=4096 \
  /data/raw/2026-10-04 /data/curated/orders/ds=2026-10-04
```

Why this design: the data is large, append-only and scanned by many downstream jobs, so columnar storage plus a scheduled batch window is both cheap and predictable.

### 2. Multi-tenant YARN cluster with department queues

A shared cluster serves ETL, BI and ad-hoc workloads. Capacity Scheduler queues guarantee 60/25/15 percent of the cluster, forbid ad-hoc jobs from consuming more than their share, and cap the ApplicationMaster resources so a runaway job cannot starve the cluster.

```xml
<property><name>yarn.scheduler.capacity.root.queues</name><value>etl,bi,adhoc</value></property>
<property><name>yarn.scheduler.capacity.root.adhoc.maximum-capacity</name><value>40</value></property>
```

Why this design: without queues, a single ad-hoc query can consume the whole cluster and delay the nightly SLA.

### 3. Migrating an on-premises cluster to object storage and Spark

An organisation moves its cold partitions to S3, keeps Iceberg as the table format, and runs Spark on Kubernetes. HDFS and YARN are retired only after every job has an owner, a runtime and a target format.

```bash
# Validate the copy before switching producers to the new path
hadoop distcp -m 40 hdfs:///data/orders s3a://acme-lake/orders
hdfs dfs -count -q -h /data/orders
aws s3 ls s3://acme-lake/orders/ --recursive --summarize --human-readable
```

Why this design: object storage removes the capacity-planning cycle and separates storage cost from compute, but only makes sense once jobs are decoupled from the old cluster.

## Hadoop exercises

### Try it 1. Size input splits for a target map count

Task: a 1.2 TiB dataset is stored as splittable Parquet with a 128 MB block size. Choose a split size that gives at least 121 maps, rounded to a whole number of blocks.

Hint: ceil division gives a raw split, rounding it up to whole blocks can overshoot and reduce the map count, so shrink by one block when that happens.

Expected result: 80 blocks (10 GB) gives only 120 maps, so the answer is 79 blocks (about 9.88 GB) and 122 maps.

In [ ]:
from math import ceil

total_bytes = 1_200 * 1024**3
block = 128 * 1024**2
target_maps = 121

raw_split = -(-total_bytes // target_maps)          # ceil division
blocks_per_split = ceil(raw_split / block)
split_size = blocks_per_split * block
map_count = ceil(total_bytes / split_size)

# Rounding up to 80 blocks overshoots and gives 120 maps, below target.
if map_count < target_maps:
    blocks_per_split -= 1
    split_size = blocks_per_split * block
    map_count = ceil(total_bytes / split_size)

assert blocks_per_split == 79
assert split_size == 79 * block
assert map_count == 122
assert map_count >= target_maps
print("split size MB:", split_size // 1024**2)
print("split size blocks:", split_size // block)
print("map count:", map_count)

### Try it 2. Compute block and replica storage from a dataset

Task: a 25 MB file and a 3 TB dataset both use a 128 MB block size with replication 3. Compute the physical bytes for each and the number of stored block replicas for the dataset.

Hint: a file of size S needs `ceil(S / block)` blocks, and each replica stores only the bytes in that block.

Expected result: the 25 MB file costs 75 MB (blocks are not padded); the 3 TB dataset needs 24,576 blocks and 9,437,184 MB (about 9.0 TiB) of physical storage.

In [ ]:
from math import ceil

MB = 1024**2
block = 128 * MB
replication = 3

small_file = 25 * MB
small_blocks = ceil(small_file / block)
small_physical = small_file * replication  # blocks are not padded
assert small_blocks == 1
assert small_physical == 75 * MB

dataset = 3 * 1024**4
blocks = ceil(dataset / block)
replicas = blocks * replication
physical = replicas * block
assert blocks == 24576
assert physical == 9_437_184 * MB
print("25 MB file physical MB:", small_physical / MB)
print("dataset blocks:", blocks, "replicas:", replicas)
print("dataset physical TiB:", round(physical / 1024**4, 2))

### Try it 3. Estimate shuffle bytes from map output and compression

Task: a job produces 250 GB of map output, a combiner removes 55 percent of the records, and map-output compression achieves 3.5x on what remains. Estimate the bytes that cross the network.

Hint: apply the combiner fraction first, then divide by the compression ratio.

Expected result: 112.5 GB after the combiner, about 32.1 GB on the wire.

In [ ]:
map_output_gb = 250.0
combiner_removed = 0.55
codec_ratio = 3.5

after_combiner = map_output_gb * (1 - combiner_removed)
on_wire = after_combiner / codec_ratio

assert abs(after_combiner - 112.5) < 1e-9
assert abs(on_wire - 32.142857142857146) < 1e-6
print("after combiner GB:", round(after_combiner, 1))
print("on wire GB:", round(on_wire, 2))

### Try it 4. Detect reducer skew from per-reducer counts

Task: 8 reducers processed these record counts. Compute the mean and the max-to-mean ratio, and list the reducers that are more than twice the mean.

Hint: a healthy distribution has a ratio near 1; values above 2 usually mean one hot key.

Expected result: mean 25,000, max 120,000, ratio 4.8, and only reducer 3 exceeds 2x the mean.

In [ ]:
counts = [120000, 8000, 9000, 12000, 7000, 9000, 11000, 24000]

mean = sum(counts) / len(counts)
worst = max(counts)
ratio = worst / mean
hot = [i for i, c in enumerate(counts) if c > 2 * mean]

assert mean == 25000.0
assert worst == 120000
assert abs(ratio - 4.8) < 1e-9
assert hot == [0]
print("mean:", mean, "max:", worst, "ratio:", round(ratio, 2))
print("skewed reducers:", hot)

### Try it 5. Compute small-file cost in NameNode memory

Task: 2 million files of 8 KB arrive in one directory. Estimate the NameNode heap they consume at roughly 150 bytes per file, and compare it with a 4 GB heap. Also compute the average block utilization at a 128 MB block size.

Hint: heap is files times bytes per file; utilization is the file size divided by the block size.

Expected result: about 286 MB of heap, roughly 7 percent of a 4 GB heap, and 0.006 percent block utilization, which is the real waste.

In [ ]:
files = 2_000_000
bytes_per_file_meta = 150
file_size = 8 * 1024
block = 128 * 1024**2
heap_gb = 4.0

meta_bytes = files * bytes_per_file_meta
meta_mb = meta_bytes / 1024**2
heap_share = meta_bytes / (heap_gb * 1024**3)
block_util = file_size / block

assert meta_mb == 286.102294921875
assert 0.06 < heap_share < 0.08
assert block_util < 0.001
print("NameNode heap MB:", round(meta_mb, 1))
print("share of 4 GB heap:", round(heap_share, 3))
print("block utilization:", round(block_util, 6))

### Try it 6. Size YARN containers from memory and vcores

Task: a node has 128 GB RAM, 32 vcores and reserves 16 GB for the OS and daemons. Map containers request 4 GB and 1 vcore; reduce containers request 8 GB and 2 vcores. How many of each fit if the node runs only that task type?

Hint: subtract the reserved memory, then take the minimum of the memory-limited and vcore-limited counts.

Expected result: 28 map containers by memory and 32 by vcores, so 28 maps; 14 reduce containers by memory and 16 by vcores, so 14 reduces.

In [ ]:
node_mem_gb = 128
reserved_gb = 16
node_vcores = 32

usable_mem_gb = node_mem_gb - reserved_gb
assert usable_mem_gb == 112

def containers(mem_gb, vcores):
    return min(usable_mem_gb // mem_gb, node_vcores // vcores)

maps = containers(4, 1)
reduces = containers(8, 2)
assert maps == 28
assert reduces == 14
print("map containers:", maps)
print("reduce containers:", reduces)

### Try it 7. Plan erasure coding savings vs replication

Task: 1 PB of cold data is stored at replication 3, then moved to RS-6-3 erasure coding (9 fragments for 6 data units). Compute the stored bytes and the monthly saving if a TB-month costs 20 dollars.

Hint: replication overhead is 3x; RS-6-3 overhead is 9/6 = 1.5x. Compare the two stored volumes and price them.

Expected result: 3,072 TB becomes 1,536 TB, a saving of 1,536 TB-month, which is 30,720 dollars per month.

In [ ]:
pb_tb = 1024
replication = 3
ec_overhead = 9 / 6
price_per_tb_month = 20.0

replicated_tb = pb_tb * replication
ec_tb = pb_tb * ec_overhead
saved_tb = replicated_tb - ec_tb
saved_usd = saved_tb * price_per_tb_month

assert replicated_tb == 3072.0
assert ec_tb == 1536.0
assert saved_tb == 1536.0
assert saved_usd == 30720.0
print("replicated TB:", replicated_tb)
print("erasure coded TB:", ec_tb)
print("monthly saving USD:", saved_usd)

### Try it 8. Choose reducer count from shuffle size and slots

Task: map output is 240 GB, you target 2 GB per reducer, the cluster offers 60 reduce containers, and the data has 90 distinct keys. Pick a reducer count that uses one wave.

Hint: reducers from size, capped by containers minus a small safety margin, then capped by distinct keys, and at least 1.

Expected result: 120 by size, capped by 57 containers, so 57 reducers.

In [ ]:
map_output_gb = 240.0
gb_per_reducer = 2.0
reduce_slots = 60
distinct_keys = 90

by_size = int(map_output_gb // gb_per_reducer)
by_slots = int(reduce_slots * 0.95)
reducers = min(by_size, by_slots, distinct_keys)
reducers = max(reducers, 1)

assert by_size == 120
assert by_slots == 57
assert reducers == 57
print("by size:", by_size, "by slots:", by_slots, "by keys:", distinct_keys)
print("reducers:", reducers)